# 小売売上データで学ぶ時系列分析入門

この notebook では `store_sales.csv` と `promotion_data.csv` を使い、小売週次売上データの確認、季節性の可視化、STL 分解、ETS / SARIMA / Prophet / LightGBM による予測、さらに特徴量設計と簡易的なアンサンブルまでを一通り確認します。

説明は実務寄りの観点で整理し、各コードセルの直前で「何を確認するのか」「なぜ必要なのか」が分かるように構成しています。

## この notebook の構成

1. データを読み込み、主キーと期間を確認する
2. 欠損週と系列の連続性を確認する
3. 単一系列を取り出して、季節性と自己相関を確認する
4. STL で系列構造を分解する
5. ETS と SARIMA で基準モデルを作る
6. Prophet で祝日と追加回帰変数を扱う
7. LightGBM でラグ特徴量と外部特徴量を扱う
8. 誤差確認、簡易チューニング、アンサンブルの流れを確認する

## 0. 準備

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
%matplotlib inline

pd.options.display.max_rows = 400
pd.options.display.max_columns = None

import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'pandas'

## 1. データの読み込み

今回使う主な列は次のとおりです。

- `store`: 店舗 ID
- `dept`: 部門 ID
- `week`: 週の起点日
- `sales`: 売上金額
- `promotion_sales`: 販促施策による売上金額

In [2]:
df_sales = pd.read_csv('data/store_sales.csv', parse_dates=['week'])
df_sales.head(2)

NameError: name 'pd' is not defined

In [3]:
df_promotion = pd.read_csv('data/promotion_data.csv', parse_dates=['week'])
df_promotion.head(2)

NameError: name 'pd' is not defined

販促データは一部の週で欠損するため、まず主キー整合を確認したうえで結合します。この notebook では欠損を「販促なし」とみなして `0` で補完しますが、実務では意味が不明な欠損を機械的に `0` 埋めしない方が安全です。

In [ ]:
df_all = pd.merge(
    df_sales,
    df_promotion,
    how='left',
    on=['store', 'dept', 'week'],
    validate='one_to_one',
)
df_all['promotion_sales'] = df_all['promotion_sales'].fillna(0)
df_all.head(2)

## 2. データのざっくり確認

まずはデータの件数、主キー、期間、重複、系列数を確認します。
この段階で系列の粒度が分かると、後続の特徴量設計やモデル選択がしやすくなります。

In [ ]:
df_sales.head(2), df_sales.shape

In [ ]:
primary_key = ['week', 'store', 'dept']
primary_key, (df_sales['week'].min(), df_sales['week'].max())

In [ ]:
df_sales.shape[0] == df_sales[['week', 'store', 'dept']].drop_duplicates().shape[0]

In [ ]:
df_sales[['store', 'dept']].drop_duplicates().shape[0]

各系列が週次で連続しているかも確認しておきます。
時系列モデルでは、欠けている週があるとラグ特徴量や周期の解釈に影響が出やすいためです。

In [ ]:
date_summary = df_sales.groupby(['store', 'dept'])['week'].agg(['min', 'max', 'nunique']).reset_index()
date_summary['missing_weeks'] = (date_summary['max'] - date_summary['min']).dt.days // 7 + 1 - date_summary['nunique']
date_summary[date_summary['missing_weeks'] > 0].head()

In [ ]:
date_summary[date_summary['missing_weeks'] > 0].shape[0]

## 3. 1 系列を例に時系列を眺める

ここからは `store=1`、`dept=1` を例にして、売上系列の形を確認します。
まずは単純な折れ線図で、全体の流れと大きな波をつかみます。

In [ ]:
df_sample_raw = (
    df_sales.query('store == 1 and dept == 1')
    .sort_values('week')
    .copy()
)

sample_missing_weeks = (
    (df_sample_raw['week'].max() - df_sample_raw['week'].min()).days // 7 + 1
    - df_sample_raw['week'].nunique()
)
print('sample_missing_weeks =', sample_missing_weeks)

df_sample = (
    df_sample_raw.set_index('week')
    .asfreq('W-MON')
    .rename_axis('week')
    .reset_index()
)

plt.figure(figsize=(12, 4))
plt.plot(df_sample['week'], df_sample['sales'], '.-')
plt.xticks(rotation=90)
plt.title('Store 1 / Dept 1 Weekly Sales')
plt.show()

In [ ]:
display(df_sample.head())
df_sample.tail(2)

グラフを見ると、年間のどこかで似たような波が繰り返されていそうに見えます。
そこで次は、年ごとに週番号をそろえて季節性を確認します。

## 4. 季節性を確認する

In [ ]:
df = df_sample.copy()
df['year'] = df['week'].dt.year
df['week_of_year'] = df['week'].dt.isocalendar().week.astype(int)

for year in df['year'].unique():
    tmp_df = df[df['year'] == year]
    plt.plot(tmp_df['week_of_year'], tmp_df['sales'], '.-', label=str(year))
plt.legend()
plt.title('Seasonal Line Plot')
plt.show()

季節性の形だけでなく、同じ週番号でどの程度ばらつくかを見るには箱ひげ図が便利です。

In [ ]:
df_boxplot = df.groupby(['week_of_year'])['sales'].agg(list).reset_index()
plt.figure(figsize=(14, 5))
plt.boxplot(df_boxplot['sales'].values, labels=df_boxplot['week_of_year'].values)
plt.xticks(rotation=90)
plt.title('Seasonal Box Plot')
plt.show()

販促の影響が強そうな時期をいったん外して、年ごとの水準差を見ることもできます。

In [ ]:
df_trend = df.copy()
df_trend = df_trend[(df_trend['week_of_year'] >= 19) & (df_trend['week_of_year'] <= 43)]
trend_boxplot = df_trend.groupby(['year'])['sales'].agg(list).reset_index()
plt.boxplot(trend_boxplot['sales'].values, labels=trend_boxplot['year'].values)
plt.title('Trend Box Plot')
plt.show()

## 5. ACF で周期性を確かめる

見た目だけでなく、自己相関から周期性を確認します。
ACF（自己相関関数）は、系列とラグ付き系列の相関を見て、どの周期で似た動きが繰り返されるかを調べる道具です。

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig = plot_acf(df_sample['sales'], lags=140)
plt.show()

fig = plot_pacf(df_sample['sales'], lags=60, method='ywm')
plt.show()

52 週付近で相関が高ければ、週次データに年周期があると考えやすくなります。

## 6. STL で系列を分解する

STL は系列を「トレンド」「季節成分」「残差」に分けて眺めるための方法です。
予測モデルではありませんが、系列の構造を理解するうえでとても役立ちます。

In [ ]:
from statsmodels.tsa.seasonal import STL

stl = STL(df_sample['sales'], period=52, robust=True)
stl_result = stl.fit()
stl_result.plot();

In [ ]:
df_stl = df_sample.copy()
df_stl['trend'] = stl_result.trend
df_stl['seasonal'] = stl_result.seasonal
df_stl['residual'] = stl_result.resid
df_stl.head()

## 7. ETS で 2 週間先を予測する

ETS は指数平滑法に基づくモデルで、比較的短めの系列でも試しやすいのが利点です。
ここでは加法トレンド・加法季節性を持つモデルとして学習します。

In [ ]:
from statsmodels.tsa.exponential_smoothing.ets import ETSModel

forecast_origin = pd.to_datetime('2012-07-30')
ets_train = df_sample[df_sample['week'] <= forecast_origin][['week', 'sales']].copy()
ets_train = ets_train.set_index('week')

ets_model = ETSModel(
    ets_train['sales'],
    error='add',
    trend='add',
    seasonal='add',
    seasonal_periods=52,
)
ets_fit = ets_model.fit()

In [ ]:
display(ets_train.head(2))
ets_train.tail()

In [ ]:
ets_train['fit_sales'] = ets_fit.fittedvalues
ets_plot = ets_train[(ets_train.index >= '2011-01-03') & (ets_train.index <= '2011-06-27')]
plt.plot(ets_plot['sales'], '.-', label='actual')
plt.plot(ets_plot['fit_sales'], '.-', label='fit')
plt.legend()
plt.xticks(rotation=90)
plt.show()

In [ ]:
h = 2
ets_forecast = ets_fit.forecast(steps=h)
ets_result = pd.DataFrame({
    'week': pd.date_range(start=ets_train.index.max() + pd.Timedelta(weeks=1), periods=h, freq='W-MON'),
    'pred_sales': ets_forecast.values,
})
ets_test = df_sample[(df_sample['week'] > forecast_origin) & (df_sample['week'] <= forecast_origin + pd.Timedelta(weeks=2))][['week', 'sales']]
ets_result.merge(ets_test, on='week', how='left')

In [ ]:
ets_result

## 8. SARIMA で 2 週間先を予測する

週次小売では年周期が強いことが多いため、この notebook では単純な ARIMA ではなく、季節成分も含めた SARIMA を基準モデルとして扱います。

まずは原系列と 1 階差分系列の自己相関を確認し、そのうえで `seasonal_order=(1, 1, 0, 52)` を持つモデルを試します。

In [ ]:
fig = plot_acf(df_sample['sales'], lags=60)
plt.show()

df_arima = df_sample.copy()
df_arima['sales_lag1'] = df_arima['sales'].shift(1)
df_arima['sales_diff'] = df_arima['sales'] - df_arima['sales_lag1']

fig = plot_acf(df_arima['sales_diff'].dropna(), lags=60)
plt.show()

fig = plot_pacf(df_arima['sales_diff'].dropna(), lags=30, method='ywm')
plt.show()

In [ ]:
df_arima[['week', 'sales', 'sales_diff']].head()

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_train = df_sample[df_sample['week'] <= forecast_origin][['week', 'sales']].copy()
sarima_train = sarima_train.set_index('week')

sarima_model = SARIMAX(
    sarima_train['sales'],
    order=(1, 1, 0),
    seasonal_order=(1, 1, 0, 52),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarima_fit = sarima_model.fit(disp=False)
sarima_fit.summary()

In [ ]:
display(sarima_train.head(2))
sarima_train.tail()

In [ ]:
sarima_forecast = sarima_fit.get_forecast(steps=2)
sarima_result = pd.DataFrame({
    'week': pd.date_range(start=sarima_train.index.max() + pd.Timedelta(weeks=1), periods=2, freq='W-MON'),
    'pred_sales': sarima_forecast.predicted_mean.values,
})
sarima_test = df_sample[(df_sample['week'] > forecast_origin) & (df_sample['week'] <= forecast_origin + pd.Timedelta(weeks=2))][['week', 'sales']]
sarima_result.merge(sarima_test, on='week', how='left')

In [ ]:
sarima_result

## 9. Prophet で祝日と販促要因を扱う

Prophet はトレンド、季節性、祝日効果、追加回帰変数を分けて扱いやすいモデルです。

ここでは `promotion_sales` を説明用の代替指標として使います。本番では販促計画値など、予測時点で既知の列へ置き換えてください。

In [ ]:
from prophet import Prophet

In [ ]:
prophet_train = (
    df_all[(df_all['week'] <= '2012-07-30') & (df_all['store'] == 1) & (df_all['dept'] == 1)]
    .copy()
    .rename(columns={'week': 'ds', 'sales': 'y'})
)

prophet_test = (
    df_all[(df_all['week'] == '2012-08-06') & (df_all['store'] == 1) & (df_all['dept'] == 1)]
    .copy()
    .rename(columns={'week': 'ds', 'sales': 'y'})
)

prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode='additive',
    changepoint_prior_scale=0.05,
    seasonality_prior_scale=10.0,
)
prophet_model.add_country_holidays(country_name='JP')
prophet_model.add_regressor('promotion_sales', standardize=True)
prophet_model.fit(prophet_train[['ds', 'y', 'promotion_sales']])

In [ ]:
display(prophet_train[['ds', 'y', 'promotion_sales']].tail())
prophet_test[['ds', 'y', 'promotion_sales']]

In [ ]:
prophet_fit = prophet_model.predict(prophet_train[['ds', 'promotion_sales']])
prophet_model.plot_components(prophet_fit)
prophet_model.plot(prophet_fit);

In [ ]:
prophet_pred = prophet_model.predict(prophet_test[['ds', 'promotion_sales']])
prophet_test['yhat'] = prophet_pred['yhat'].values
prophet_test[['ds', 'y', 'promotion_sales', 'yhat']]

### Prophet の簡易チューニング

実務では `changepoint_prior_scale` や `seasonality_mode` を 1 回で決め打ちせず、rolling origin で比較してから選びます。

ここでは `promotion_sales` を説明用の代替指標として使い、複数候補を比較する最小例を示します。

In [ ]:
def wape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return 100 * np.sum(np.abs(y_true - y_pred)) / np.maximum(np.sum(np.abs(y_true)), eps)

prophet_df = (
    df_all.query('store == 1 and dept == 1')
    .sort_values('week')
    .rename(columns={'week': 'ds', 'sales': 'y'})
    .copy()
)
prophet_df['promotion_plan_proxy'] = prophet_df['promotion_sales']

cutoffs = pd.to_datetime([
    '2011-12-26',
    '2012-03-26',
    '2012-07-30',
])

param_grid = [
    {
        'changepoint_prior_scale': 0.03,
        'seasonality_prior_scale': 5.0,
        'seasonality_mode': 'additive',
    },
    {
        'changepoint_prior_scale': 0.10,
        'seasonality_prior_scale': 10.0,
        'seasonality_mode': 'additive',
    },
    {
        'changepoint_prior_scale': 0.10,
        'seasonality_prior_scale': 10.0,
        'seasonality_mode': 'multiplicative',
    },
]

prophet_cv_rows = []

for params in param_grid:
    fold_scores = []

    for cutoff in cutoffs:
        train = prophet_df.loc[
            prophet_df['ds'] <= cutoff,
            ['ds', 'y', 'promotion_plan_proxy'],
        ]
        valid = prophet_df.loc[
            (prophet_df['ds'] > cutoff)
            & (prophet_df['ds'] <= cutoff + pd.Timedelta(weeks=4)),
            ['ds', 'y', 'promotion_plan_proxy'],
        ]

        if valid.empty:
            continue

        model = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=False,
            daily_seasonality=False,
            changepoint_prior_scale=params['changepoint_prior_scale'],
            seasonality_prior_scale=params['seasonality_prior_scale'],
            seasonality_mode=params['seasonality_mode'],
        )
        model.add_country_holidays(country_name='JP')
        model.add_regressor('promotion_plan_proxy', standardize=True)
        model.fit(train)

        pred = model.predict(valid[['ds', 'promotion_plan_proxy']])
        fold_scores.append(wape(valid['y'], pred['yhat']))

    prophet_cv_rows.append({
        **params,
        'mean_wape': np.mean(fold_scores),
        'std_wape': np.std(fold_scores),
    })

prophet_cv_result = pd.DataFrame(prophet_cv_rows).sort_values('mean_wape')
prophet_cv_result

### Prophet に外部要因を入れる最小例

教材データには気象や人流の実データが含まれていないため、この例では週番号から説明用の外部特徴量を作っています。

実務では、この部分を気象予報、販促計画、人流データなどの週次テーブルへ置き換え、`add_regressor()` で同じようにモデルへ渡します。

In [ ]:
calendar_helper = prophet_df[['ds']].drop_duplicates().copy()
calendar_helper['week_of_year'] = calendar_helper['ds'].dt.isocalendar().week.astype(int)
calendar_helper['month'] = calendar_helper['ds'].dt.month

weather_weekly = calendar_helper[['ds']].copy()
weather_weekly['avg_temp_forecast'] = 15 + 10 * np.sin(2 * np.pi * calendar_helper['week_of_year'] / 52)
weather_weekly['rainfall_forecast'] = 5 + 3 * np.cos(2 * np.pi * calendar_helper['week_of_year'] / 52)

promo_plan_weekly = calendar_helper[['ds']].copy()
promo_plan_weekly['promo_flag_plan'] = calendar_helper['week_of_year'].isin([1, 13, 26, 39, 48]).astype(int)
promo_plan_weekly['discount_rate_plan'] = promo_plan_weekly['promo_flag_plan'] * 0.10

foot_traffic_weekly = calendar_helper[['ds']].copy()
foot_traffic_weekly['foot_traffic_data'] = (
    100
    + 12 * np.sin(2 * np.pi * calendar_helper['week_of_year'] / 52)
    + 4 * calendar_helper['month'].isin([12, 1]).astype(int)
)

prophet_exog = prophet_df[['ds', 'y']].copy()
prophet_exog = (
    prophet_exog
    .merge(weather_weekly, on='ds', how='left', validate='one_to_one')
    .merge(promo_plan_weekly, on='ds', how='left', validate='one_to_one')
    .merge(foot_traffic_weekly, on='ds', how='left', validate='one_to_one')
)

regressor_cols = [
    'avg_temp_forecast',
    'rainfall_forecast',
    'discount_rate_plan',
    'promo_flag_plan',
    'foot_traffic_data',
]

prophet_exog[regressor_cols] = prophet_exog[regressor_cols].fillna(0)

prophet_exog_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode='multiplicative',
)
prophet_exog_model.add_country_holidays(country_name='JP')

for col in regressor_cols:
    prophet_exog_model.add_regressor(col, standardize=True)

prophet_exog_model.fit(prophet_exog[['ds', 'y'] + regressor_cols])
prophet_exog[['ds', 'y'] + regressor_cols].head()

## 10. LightGBM でラグ特徴量と説明変数を扱う

LightGBM では系列そのものを渡すのではなく、ラグ特徴量、移動平均、カレンダー特徴量、販促の代替列などを表形式データへ落とし込みます。

ここでは、教材データだけで動く範囲で、初学者が追いやすい基本形を作ります。

In [ ]:
import lightgbm as lgb
from lightgbm import LGBMRegressor

In [ ]:
lgb_sample = (
    df_all[(df_all['store'] == 1) & (df_all['dept'] == 1)]
    .sort_values('week')
    .copy()
)

lgb_sample['week_of_year'] = lgb_sample['week'].dt.isocalendar().week.astype(int)
lgb_sample['month'] = lgb_sample['week'].dt.month
lgb_sample['week_sin'] = np.sin(2 * np.pi * lgb_sample['week_of_year'] / 52)
lgb_sample['week_cos'] = np.cos(2 * np.pi * lgb_sample['week_of_year'] / 52)

lgb_sample['sales_lag_1'] = lgb_sample['sales'].shift(1)
lgb_sample['sales_lag_4'] = lgb_sample['sales'].shift(4)
lgb_sample['sales_lag_52'] = lgb_sample['sales'].shift(52)
lgb_sample['promo_lag_1'] = lgb_sample['promotion_sales'].shift(1)
lgb_sample['promo_lag_52'] = lgb_sample['promotion_sales'].shift(52)
lgb_sample['sales_roll_mean_4'] = lgb_sample['sales'].shift(1).rolling(4).mean()
lgb_sample['sales_roll_mean_13'] = lgb_sample['sales'].shift(1).rolling(13).mean()

# 教材では説明用に当週の販促列から代替列を作る。
# 実務では販促計画表など、予測時点で既知の列へ置き換える。
lgb_sample['promotion_plan_proxy'] = lgb_sample['promotion_sales']

feature_cols = [
    'week_sin',
    'week_cos',
    'month',
    'sales_lag_1',
    'sales_lag_4',
    'sales_lag_52',
    'promo_lag_1',
    'promo_lag_52',
    'sales_roll_mean_4',
    'sales_roll_mean_13',
    'promotion_plan_proxy',
]

lgb_sample = lgb_sample.dropna(subset=feature_cols)

x_train = lgb_sample.loc[lgb_sample['week'] <= '2012-07-30', feature_cols]
y_train = lgb_sample.loc[lgb_sample['week'] <= '2012-07-30', 'sales']
x_test = lgb_sample.loc[lgb_sample['week'] == '2012-08-06', feature_cols]
y_test = lgb_sample.loc[lgb_sample['week'] == '2012-08-06', 'sales']

In [ ]:
display(lgb_sample[['week'] + feature_cols + ['sales']].head())
lgb_sample.shape

In [ ]:
lgb_model = LGBMRegressor(
    objective='regression',
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=42,
)
lgb_model.fit(x_train, y_train)
y_pred = lgb_model.predict(x_test)
pd.DataFrame({'y_test': y_test.values, 'y_pred': y_pred})

複数系列をまとめるときは、系列ごとにラグを計算することが重要です。`groupby(['store', 'dept'])` を付けずに `shift()` すると、別系列の値が混ざります。

In [ ]:
lgb_all = df_all.sort_values(['store', 'dept', 'week']).copy()

lgb_all['sales_lag_1'] = lgb_all.groupby(['store', 'dept'])['sales'].shift(1)
lgb_all['promo_lag_1'] = lgb_all.groupby(['store', 'dept'])['promotion_sales'].shift(1)

lgb_all[['store', 'dept', 'week', 'sales', 'sales_lag_1', 'promo_lag_1']].head()

In [ ]:
weekly_calendar = lgb_sample[['week']].drop_duplicates().copy()
weekly_calendar['week_of_year'] = weekly_calendar['week'].dt.isocalendar().week.astype(int)
weekly_calendar['month'] = weekly_calendar['week'].dt.month
weekly_calendar['promo_flag_plan'] = weekly_calendar['week_of_year'].isin([1, 13, 26, 39, 48]).astype(int)
weekly_calendar['discount_rate_plan'] = weekly_calendar['promo_flag_plan'] * 0.10
weekly_calendar['avg_temp_forecast'] = 15 + 10 * np.sin(2 * np.pi * weekly_calendar['week_of_year'] / 52)
weekly_calendar['rainfall_forecast'] = 5 + 3 * np.cos(2 * np.pi * weekly_calendar['week_of_year'] / 52)
weekly_calendar['foot_traffic_data'] = (
    100
    + 12 * np.sin(2 * np.pi * weekly_calendar['week_of_year'] / 52)
    + 4 * weekly_calendar['month'].isin([12, 1]).astype(int)
)
weekly_calendar['inventory_on_hand'] = 400 + 40 * np.cos(2 * np.pi * weekly_calendar['week_of_year'] / 52)

lgb_feature_table = (
    lgb_sample[['week', 'sales']].copy()
    .merge(
        weekly_calendar[
            [
                'week',
                'promo_flag_plan',
                'discount_rate_plan',
                'avg_temp_forecast',
                'rainfall_forecast',
                'foot_traffic_data',
                'inventory_on_hand',
            ]
        ],
        on='week',
        how='left',
        validate='one_to_one',
    )
    .assign(
        sales_lag_1=lambda d: d['sales'].shift(1),
        sales_lag_52=lambda d: d['sales'].shift(52),
        sales_roll_mean_4=lambda d: d['sales'].shift(1).rolling(4).mean(),
        week_sin=lambda d: np.sin(2 * np.pi * d['week'].dt.isocalendar().week.astype(int) / 52),
        week_cos=lambda d: np.cos(2 * np.pi * d['week'].dt.isocalendar().week.astype(int) / 52),
        discount_x_foot_traffic=lambda d: d['discount_rate_plan'] * d['foot_traffic_data'],
        temp_x_promo=lambda d: d['avg_temp_forecast'] * d['promo_flag_plan'],
    )
)

display(lgb_feature_table.head())

### 作成した外部特徴量をそのままモデルへ渡す例

特徴量テーブルを作っただけでは予測モデルにはなりません。ここでは、生成した外部特徴量を `LightGBM` へ渡し、1 週先予測へ使う最小例を確認します。

In [ ]:
external_feature_cols = [
    'sales_lag_1',
    'sales_lag_52',
    'sales_roll_mean_4',
    'promo_flag_plan',
    'discount_rate_plan',
    'avg_temp_forecast',
    'rainfall_forecast',
    'foot_traffic_data',
    'inventory_on_hand',
    'week_sin',
    'week_cos',
    'discount_x_foot_traffic',
    'temp_x_promo',
]

lgb_feature_table = lgb_feature_table.dropna(subset=external_feature_cols + ['sales'])

X_train_ext = lgb_feature_table.loc[lgb_feature_table['week'] <= '2012-07-30', external_feature_cols]
y_train_ext = lgb_feature_table.loc[lgb_feature_table['week'] <= '2012-07-30', 'sales']
X_test_ext = lgb_feature_table.loc[lgb_feature_table['week'] == '2012-08-06', external_feature_cols]
y_test_ext = lgb_feature_table.loc[lgb_feature_table['week'] == '2012-08-06', 'sales']

lgb_external_model = LGBMRegressor(
    objective='regression',
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=42,
)
lgb_external_model.fit(X_train_ext, y_train_ext)
external_pred = lgb_external_model.predict(X_test_ext)

pd.DataFrame({'y_test': y_test_ext.values, 'y_pred': external_pred})

## 11. 実務上の補足: 指標、リーク確認、調整、アンサンブル

ここでは本文で追加した重要論点のうち、初学者が特に見落としやすい項目をコードで確認します。

- 誤差指標をまとめて計算する
- 予測時点で未知の列を混ぜていないか確認する
- LightGBM の主要パラメータをどう比較するか
- 複数モデルの予測をどう重み付き平均するか
- 異常値をどう見つけるか
- 間欠需要をどう見分けるか

In [ ]:
pred_table = pd.DataFrame(
    {
        'week': [pd.Timestamp('2012-08-06')],
        'y_true': prophet_test['y'].values,
        'ets_pred': ets_result.loc[
            ets_result['week'] == pd.Timestamp('2012-08-06'),
            'pred_sales',
        ].values,
        'sarima_pred': sarima_result.loc[
            sarima_result['week'] == pd.Timestamp('2012-08-06'),
            'pred_sales',
        ].values,
        'prophet_pred': prophet_test['yhat'].values,
        'lgb_pred': y_pred,
    }
)

model_cols = ['ets_pred', 'sarima_pred', 'prophet_pred', 'lgb_pred']

score_df = pd.DataFrame(
    {
        'model': model_cols,
        'wape': [wape(pred_table['y_true'], pred_table[col]) for col in model_cols],
    }
)
score_df['raw_weight'] = 1 / score_df['wape']
score_df['weight'] = score_df['raw_weight'] / score_df['raw_weight'].sum()
weight_map = dict(zip(score_df['model'], score_df['weight']))

pred_table['ensemble_pred'] = sum(
    pred_table[col] * weight_map[col]
    for col in model_cols
)

score_df, pred_table

In [ ]:
tune_candidates = [
    {'num_leaves': 15, 'min_child_samples': 40},
    {'num_leaves': 31, 'min_child_samples': 20},
    {'num_leaves': 63, 'min_child_samples': 10},
]

valid_start = pd.to_datetime('2012-07-02')
valid_end = pd.to_datetime('2012-07-30')

train_mask = lgb_sample['week'] < valid_start
valid_mask = (lgb_sample['week'] >= valid_start) & (lgb_sample['week'] <= valid_end)

X_train_tune = lgb_sample.loc[train_mask, feature_cols]
y_train_tune = lgb_sample.loc[train_mask, 'sales']
X_valid_tune = lgb_sample.loc[valid_mask, feature_cols]
y_valid_tune = lgb_sample.loc[valid_mask, 'sales']

lgb_cv_rows = []

for candidate in tune_candidates:
    model = LGBMRegressor(
        objective='regression',
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=candidate['num_leaves'],
        min_child_samples=candidate['min_child_samples'],
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
    )
    model.fit(
        X_train_tune,
        y_train_tune,
        eval_set=[(X_valid_tune, y_valid_tune)],
        eval_metric='l1',
        callbacks=[lgb.early_stopping(100, verbose=False)],
    )
    pred = model.predict(X_valid_tune, num_iteration=model.best_iteration_)
    lgb_cv_rows.append({
        **candidate,
        'best_iteration': model.best_iteration_,
        'valid_wape': wape(y_valid_tune, pred),
    })

pd.DataFrame(lgb_cv_rows).sort_values('valid_wape')

In [ ]:
forbidden_future_actuals = {
    'promotion_sales',
    'actual_customer_count',
    'actual_stockout_count',
    'realized_price',
}

def assert_future_known(feature_cols, forbidden_cols):
    leaked = sorted(set(feature_cols) & set(forbidden_cols))
    if leaked:
        raise ValueError(f'情報漏洩の疑いがある列: {leaked}')

candidate_feature_cols = [
    'sales_lag_1',
    'sales_lag_52',
    'promo_flag_plan',
    'discount_rate_plan',
    'avg_temp_forecast',
    'foot_traffic_data',
]

assert_future_known(candidate_feature_cols, forbidden_future_actuals)
print('リーク検査を通過しました')

### 指標・異常値・間欠需要の確認

誤差指標は 1 つだけでなく複数並べて見る方が安全です。また、小売時系列では異常値と間欠需要の有無がモデル選択と評価指標に直接影響します。

ここでは、予測結果の代表指標をまとめて計算し、あわせて異常値候補と 0 売上週の比率を確認します。

In [ ]:
def regression_metrics(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), eps))) * 100
    wape_value = np.sum(np.abs(y_true - y_pred)) / np.maximum(np.sum(np.abs(y_true)), eps) * 100
    smape = np.mean(
        2 * np.abs(y_true - y_pred) / np.maximum(np.abs(y_true) + np.abs(y_pred), eps)
    ) * 100

    return {
        'MAE': mae,
        'RMSE': rmse,
        'MAPE(%)': mape,
        'WAPE(%)': wape_value,
        'sMAPE(%)': smape,
    }

metrics_df = pd.DataFrame(
    [
        {'model': col, **regression_metrics(pred_table['y_true'], pred_table[col])}
        for col in model_cols + ['ensemble_pred']
    ]
)
metrics_df

In [ ]:
df_anomaly = df_sample.copy()
df_anomaly['rolling_median_13'] = df_anomaly['sales'].shift(1).rolling(13, min_periods=4).median()
df_anomaly['rolling_iqr_13'] = (
    df_anomaly['sales'].shift(1).rolling(13, min_periods=4).quantile(0.75)
    - df_anomaly['sales'].shift(1).rolling(13, min_periods=4).quantile(0.25)
)
df_anomaly['anomaly_flag'] = (
    (df_anomaly['sales'] > df_anomaly['rolling_median_13'] + 3 * df_anomaly['rolling_iqr_13'])
    | (df_anomaly['sales'] < df_anomaly['rolling_median_13'] - 3 * df_anomaly['rolling_iqr_13'])
).fillna(False)

df_anomaly.loc[df_anomaly['anomaly_flag'], ['week', 'sales']].head()

In [ ]:
df_intermittent = df_sample.copy()
df_intermittent['is_positive_sale'] = (df_intermittent['sales'] > 0).astype(int)

zero_ratio = 1 - df_intermittent['is_positive_sale'].mean()
avg_positive_sale = (
    df_intermittent.loc[
        df_intermittent['is_positive_sale'] == 1,
        'sales',
    ].mean()
)

pd.Series(
    {
        'zero_week_ratio': zero_ratio,
        'average_positive_sale': avg_positive_sale,
    }
)

## まとめ

この notebook では、小売売上データを題材に、データ確認から季節性の把握、STL 分解、ETS / SARIMA / Prophet / LightGBM による予測、そして簡易的なチューニングとアンサンブルまでを確認しました。

特に重要なのは、モデルを比べる前に、主キー、欠損週、将来既知の変数、評価の切り方を先に固めることです。小売時系列では、モデル選択そのもの以上に、特徴量設計と情報漏洩の回避が精度を左右します。